# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
## I was really struggling to get the original path to work
## I looked online, the easiest way for me to do it was to download the file directly
## and to just change the path to that file.
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))


original_length = len(cat)
print(f'{original_length} rows, {cat["sku"].nunique()} distinct SKUs')
cat.head()

45 rows, 45 distinct SKUs


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
rows_with_pop = cat['product_name'].str.lower().str.contains('pop', na=False)
pop_rows = cat[rows_with_pop]
print('rows with "pop" in the name:', len(pop_rows))
display(pop_rows)

rows_with_poptart = cat['product_name'].str.contains(r'pop[\s\-]?tart', case=False, na=False, regex=True)
print('rows that are actually Pop-Tarts:', rows_with_poptart.sum())
print('"pop" matches that are NOT Pop-Tarts:')
display(cat[rows_with_pop & ~rows_with_poptart])

rows with "pop" in the name: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


rows that are actually Pop-Tarts: 4
"pop" matches that are NOT Pop-Tarts:


,sku,product_name,category,subcategory,unit_cost,supplier,package_size


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
poptarts = cat[rows_with_poptart]
is_strawberry = poptarts['product_name'].str.contains('straw', case=False, na=False)

straw = poptarts[is_strawberry]
not_straw = poptarts[~is_strawberry]

print('STRAWBERRY Pop-Tart SKUs (these all mean one product):')
display(straw[['sku', 'product_name', 'category']])
print('Strawberry SKUs:', sorted(straw['sku'].unique()))

print('\nPop-Tart rows that are NOT strawberry (keep separate):')
display(not_straw[['sku', 'product_name', 'category']])

STRAWBERRY Pop-Tart SKUs (these all mean one product):


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast


Strawberry SKUs: ['POPTART-STRAW', 'PT-12', 'PT_STRAWBERRY', 'SKU#459812']

Pop-Tart rows that are NOT strawberry (keep separate):


,sku,product_name,category


**Pop-Tart rows that are a different product:** None of them. All of the pop tart rows in the catalog are strawberry, so nothing has to be kept separate. If another flavor like blueberry was to show up it would have to stay a different product, however given the current conditions none of them are a different product.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
CANON_STRAWBERRY = 'POPTART-STRAW'

CANON = {s: CANON_STRAWBERRY for s in straw['sku'].unique()}

clean = cat.copy()
clean['canonical_sku'] = clean['sku'].map(CANON).fillna(clean['sku'])

print('mapping:', CANON)
print('distinct SKUs before:', clean['sku'].nunique(),
      '-> after:', clean['canonical_sku'].nunique())

print('rows that .map() WITHOUT the fillna piece would have turned into NaN:',
      clean['sku'].map(CANON).isna().sum(), 'of', len(clean))

mapping: {'PT-12': 'POPTART-STRAW', 'SKU#459812': 'POPTART-STRAW', 'POPTART-STRAW': 'POPTART-STRAW', 'PT_STRAWBERRY': 'POPTART-STRAW'}
distinct SKUs before: 45 -> after: 42
rows that .map() WITHOUT the fillna piece would have turned into NaN: 41 of 45


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print('BEFORE:')
print(clean['category'].value_counts(dropna=False).to_string())

clean['category'] = clean['category'].str.strip().str.title()
print('\nAFTER strip + title-case:')
print(clean['category'].value_counts(dropna=False).to_string())
# After stripping and title-casing, five pairs are still two labels for one category.
# Emergency and Emergency Supplies, Breakfast and Breakfast Foods, Grocery and Groceries,
# Baby and Baby Products, and Alcoholic Beverages and Alcohol. Since they are different words, title casing doesn't fix it.
# I created a category_map mapping sequence for if categories need to be further
# Condensed in the future, as they are pairs that are the same category but not the same string.
CATEGORY_MAP = {
    'Breakfast Foods':    'Breakfast',
    'Emergency Supplies': 'Emergency',
    'Alcohol':            'Alcoholic Beverages',
    'Groceries':          'Grocery',
    'Baby Products':      'Baby',
}


BEFORE:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1

AFTER strip + title-case:
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods       

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
clean['normalized'] = (clean['product_name'].str.lower()
                       .str.replace(r'[^a-z0-9 ]', ' ', regex=True)
                       .str.split()
                       .apply(lambda w: ' '.join(sorted(w))))

candidates = (clean.groupby('normalized')['sku']
              .agg(list)
              .loc[lambda s: s.str.len() > 1])

print(f'{len(candidates)} candidate groups, covering {candidates.str.len().sum()} SKUs')
candidates

14 candidate groups, covering 30 SKUs


,sku
normalized,
24 beer light pack,"[BEER-24, LTBEER24]"
4 aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
aid first kit,"[FAID-01, FIRSTAID-KIT]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
number_of_null = clean['canonical_sku'].isna().sum()
assert number_of_null == 0
print(f'{number_of_null} rows lost their SKU in the mapping')

assert len(clean) == original_length, f'row count changed: {original_length} -> {len(clean)}'


0 rows lost their SKU in the mapping


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) The first product family that is still fragmented is batteries. When looking through the raw file, we see that there is a battery SKU named BATT-100 that did not get picked up in the candidate grouping.  The second one is bottled water. In the raw data we see that bottled water is listed under three different SKUs. However, it was not grouped together due to spelling, varying on how names are written. I would group them together by mapping all of their SKUs to the same canonical SKU.

b) Consolidation stops being about cleaning when you go from just converting strings to the same label to seeing if two products are actually the same thing. We know that "Strawberry Pop-Tart" and "Pop-Tart Strawberry" are likely the same. However, if the product name is just "Pop-Tart", we have no idea what flavor it is. If it is "Breakfast-Pastry", we don't know if it is even a pop tart. That is when consolidation stops being about data cleaning. In the file, we see the case of STREAM-01 and ROKU-STD. They are both related to Roku, however we do not know if the streaming device and the roku streaming stick are different. I would ask the manager of the store who oversees the electronics sales, as they would likely know what they are actually selling.